# Custom paddock boundaries

If you have field boundaries from QGIS, a cadastral layer, an earlier ground survey, or any other source, you can skip SAM segmentation and have PaddockTS run the per-paddock time series, plots, videos, and PDF report against *your* polygons.

Supported formats: GeoPackage (`.gpkg`), Shapefile (`.shp`), GeoJSON (`.geojson` / `.json`). The file just needs a polygon column; if a `paddock` column is missing, it's added as a 1-based integer ID. If `area_ha` or `compactness` are missing, they're computed from the geometry.

In [ ]:
import geopandas as gpd

# This notebook uses the bundled artifact at the repo root. Replace with
# your own path to try on your data.
paddocks_fp = "../artifacts/PaddockSet1.gpkg"

paddocks = gpd.read_file(paddocks_fp)
print(f"{len(paddocks)} paddocks, columns: {list(paddocks.columns)}")
paddocks.head()

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 8))
paddocks.plot(facecolor="none", edgecolor="red", linewidth=1, ax=ax)
ax.set_title(f"{len(paddocks)} user-provided paddocks")
ax.set_aspect("equal")
plt.show()

## Build a `Query` from the paddocks file

`Query.build_from_paddocks` takes the envelope of all paddock geometries as the AOI, so the Sentinel-2 download covers exactly the area you care about. The file is reprojected to EPSG:4326 if needed; the resulting bbox is `[W, S, E, N]` in decimal degrees.

Pass `label_col=` if your file has a human-readable name column you want surfaced in plots (instead of the numeric ID).

In [ ]:
from datetime import date
from PaddockTS.query import Query

query = Query.build_from_paddocks(
    paddocks_filepath=paddocks_fp,
    start=date(2024, 1, 1),
    end=date(2024, 12, 31),
    stub="PaddockSet1",
    # label_col="paddock_name",  # uncomment if your file has a name column
)
print(query)

## Run the full pipeline against your paddocks

Three things to pass through to `get_outputs`:

- `paddocks_filepath=` — the file to read.
- `skip_sam=True` — don't run SAM segmentation at all. Required if you only want user-paddock outputs.
- `label_col=` — column to use for paddock labels in calendars / videos / phenology plots. Defaults to the numeric `paddock` ID.

Or drop `skip_sam=True` to run **both** SAM and your paddocks side by side — useful for comparison. The PDF report then includes calendar + phenology sections for each.

In [ ]:
from PaddockTS.get_outputs import get_outputs

get_outputs(
    query,
    paddocks_filepath=paddocks_fp,
    skip_sam=True,
    label_col="paddock",
)

## Where the outputs went

User-paddock outputs are keyed by the **paddocks file stem** (here `PaddockSet1`), not the SAM stem — so they don't collide with a SAM run on the same query.

In [ ]:
from pathlib import Path

print(f"All outputs under {query.out_dir}:\n")
for p in sorted(Path(query.out_dir).iterdir()):
    print(f"  {p.name}")

## Direct call: skip the orchestrator entirely

Any individual stage accepts a `paddocks_filepath` argument so you can call it without `get_outputs`. The per-paddock TS, for example:

In [ ]:
from PaddockTS.Phenology.make_paddock_time_series import make_paddock_time_series

ts = make_paddock_time_series(query, paddocks_filepath=paddocks_fp)
print(ts)

In [ ]:
# Plot NDVI through the year for the first few paddocks
fig, ax = plt.subplots(figsize=(10, 4))
for p in ts.paddock.values[:5]:
    ts.NDVI.sel(paddock=p).plot(ax=ax, label=p, marker=".")
ax.set_title("NDVI per paddock (user boundaries)")
ax.set_ylabel("NDVI")
ax.legend(loc="upper right")
plt.tight_layout()
plt.show()

## See also

- [`01_quickstart.ipynb`](01_quickstart.ipynb) — auto-segmented (SAM) end-to-end.
- [`02_pipeline_stages.ipynb`](02_pipeline_stages.ipynb) — calling stages individually with explanations.
- [`Query.build_from_paddocks` docs](https://thestochasticman.github.io/paddock-ts-local/api/query/) — full constructor reference.